### KMeans clustering

In [4]:
# Izbor optimalnog broja klastera pomoću Elbow metode i Silhouette analize, kao i vizualizacija rezultata klasterovanja. 
# Takođe, uključuje kod za čuvanje rezultata klasterovanja u .npy formatu radi efikasnijeg učitavanja i rada sa modelima.

# Ucitavanje potrebnih podataka
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import seaborn as sns
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

datasets = {
    "Scaled dataset": "../../../data/reduced/X_scaled.npy",
    "PCA dataset": "../../../data/reduced/X_pca.npy"
}

outputDirectory = "../../../results/kmeans"

In [ ]:
def elbow_method(X, name):
    K = list(range(2, 15))
    inertia = []

    for k in K:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        inertia.append(kmeans.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(K, inertia, marker='o')
    plt.xlabel('Number of clusters')
    plt.ylabel('Inertia')
    plt.title(f'Elbow Method ({name})')
    plt.grid()
    plt.tight_layout()
    safe_dataset_name = name.replace(" ", "_").lower()
    plt.savefig(f"{outputDirectory}/elbow_{safe_dataset_name}.png", dpi=200)
    plt.close()

    return K[np.argmax(np.diff(inertia, 2)) + 2]

In [ ]:
data = pd.read_csv("../../../data/preprocessed/preprocessed_dataset.csv")
y = data["expression"].astype("category").cat.codes

results = []
for dataset_name, X_data in datasets.items():
    X = np.load(X_data)

    safe_dataset_name = dataset_name.replace(" ", "_").lower()

    k = elbow_method(X, dataset_name)
    print(f"Selected number of clusters based on elbow method for {dataset_name}: {k}")
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)

    row = {
        "Dataset": dataset_name,
        "Algorithm": "KMeans",
        "K": k,
        "Silhouette": silhouette_score(
            X,
            labels,
            sample_size=min(3000, X.shape[0]),
            random_state=42
        ),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "ARI": adjusted_rand_score(y, labels),
        "NMI": normalized_mutual_info_score(y, labels)
    }
    results.append(row)

    if dataset_name == "PCA dataset":
        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        X_tsne = tsne.fit_transform(X)

        plt.figure(figsize=(6,5))
        plt.scatter(X_tsne[:,0], X_tsne[:,1], c=labels, s=5, cmap="tab10")
        plt.title(f"KMeans TSNE ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"{outputDirectory}/kmTSNE_{safe_dataset_name}.png", dpi=200)
        plt.close()

        plt.figure(figsize=(6,5))
        plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, s=5, cmap="tab10")
        plt.title(f"True classes TSNE ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"{outputDirectory}/trueTSNE_{safe_dataset_name}.png", dpi=200)
        plt.close()

    # tabela cluster vs true class sa procentima, čuvanje u CSV formatu i štampanje
    cluster_vs_class = pd.crosstab(
        pd.Series(labels, name="cluster"),
        pd.Series(data["expression"], name="expression"),
        normalize="index"
    )

    cluster_vs_class = cluster_vs_class.round(4)
    safe_dataset_name = dataset_name.replace(" ", "_").lower()
    cluster_vs_class.to_csv(f"{outputDirectory}/cluster_vs_class_{safe_dataset_name}.csv")
    print(cluster_vs_class)

    # heatmap
    plt.figure(figsize=(8,6))
    sns.heatmap(cluster_vs_class, annot=True, cmap="Blues")
    plt.title(f"Cluster vs True Class ({dataset_name})")
    plt.tight_layout()
    plt.savefig(f"{outputDirectory}/cluster_vs_class_heatmap_{safe_dataset_name}.png", dpi=200)
    plt.close()

results_df = pd.DataFrame(results)
results_df = results_df.round({
    "Silhouette": 4,
    "Davies_Bouldin": 4,
    "ARI": 4,
    "NMI": 4
})

results_df.to_csv(f"{outputDirectory}/kmeans_results_summary.csv", index=False)
print(results_df)

Selected number of clusters based on elbow method for Scaled dataset: 4
expression  affirmative  conditional  doubt_question  emphasis  negative  \
cluster                                                                    
0                0.0858       0.1890          0.0080    0.1904    0.0235   
1                0.0000       0.1939          0.3729    0.1107    0.0308   
2                0.0929       0.1236          0.0585    0.0866    0.1287   
3                0.0727       0.0879          0.1557    0.0000    0.1606   

expression  relative  topics  wh_question  yn_question  
cluster                                                 
0             0.2662  0.2290       0.0034       0.0048  
1             0.0385  0.0709       0.0081       0.1742  
2             0.1543  0.1389       0.1127       0.1038  
3             0.0624  0.0184       0.2131       0.2291  
Selected number of clusters based on elbow method for PCA dataset: 4
expression  affirmative  conditional  doubt_question  emphas

##### KMeans algoritam je primenjen na scaled dataset i PCA dataset. Broj klastera izabran je pomoću Elbow metode, pri čemu je za oba skupa izabrano K = 4. PCA dataset ostvaruje nešto bolju Silhouette vrednost i niži Davies-Bouldin indeks, što ukazuje na blago bolju geometrijsku separaciju klastera u redukovanom prostoru.

##### Međutim, vrednosti ARI i NMI su niske, što pokazuje da dobijeni klasteri nisu dobro usklađeni sa stvarnim klasama gramatičkih izraza. Analiza raspodele stvarnih klasa po klasterima pokazuje da su klasteri mešoviti i da ne odgovaraju jednoznačno pojedinačnim izrazima. Zbog toga se KMeans može smatrati korektno primenjenim, ali ograničeno uspešnim za ovaj skup podataka.